# Trabalho Prático 2 — Pré-processamento e Análise Preditiva
## Notebook 2: Pré-processamento (Etapas 6 a 10)

**Aluno:** João Victor Borges Carvalho  
**Base:** Breast Cancer Wisconsin (Original)  
**Abordagem:** Tudo do zero com Python puro + math  

Segundo notebook! Agora que já explorei os dados, é hora de preparar o terreno pros algoritmos. Aqui eu faço a limpeza completa: removo duplicatas, analiso desbalanceamento, trato outliers (ou não), normalizo e aplico correlação de Pearson pra ver se dá pra reduzir a dimensionalidade.

Cada decisão aqui impacta diretamente o desempenho dos modelos no Notebook 3, então nada de aplicar transformação no automático — preciso justificar cada escolha.

## Configuração e Imports

> ⚠️ Execute o **Notebook 1** antes pra ter as variáveis `X_train`, `X_test`, `y_train`, `y_test` em memória.

Se já executou o Notebook 1, pode recarregar os dados com a célula abaixo — o pipeline é reproduzido do zero. Isso é útil se você quiser pular direto pra cá.

In [ ]:
import os
import sys
from collections import Counter

PROJECT_DIR = os.path.dirname(os.path.abspath(''))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from utils import (
    parse_dataset, extract_column, count_target, calc_mode,
    calc_mean, calc_median, calc_variance, calc_std, calc_min, calc_max, calc_range,
    calc_percentile, calc_quartiles, calc_iqr,
    format_percent,
    stratified_split,
    remove_constant_columns, remove_duplicate_rows,
    detect_outliers_iqr,
    impute_mode,
    normalize_minmax,
    feature_target_correlations,
    calc_correlation,
)

DATA_PATH = os.path.join('data', 'breast_cancer_wisconsin.csv')
FEATURE_MIN = 1
FEATURE_MAX = 10

# Recarrega os dados e reproduz a divisão
header, X, y, mask = parse_dataset(DATA_PATH)
X_valid = [X[i] for i in range(len(X)) if mask[i]]
y_valid = [y[i] for i in range(len(y)) if mask[i]]
X_train, X_test, y_train, y_test = stratified_split(X_valid, y_valid, test_ratio=0.2, seed=42)

feature_names = list(header[:-1])

print(f'✅ Dados carregados: {len(y_train)} treino, {len(y_test)} teste')

## Etapa 6 — Eliminação de Atributos e Exemplos

Primeiro verifico se tem coluna constante (variância zero). Se uma coluna tem sempre o mesmo valor, ela é totalmente inútil pra classificação — não ajuda em nada a diferenciar benigno de maligno. Depois removo exemplos duplicados, que podem inflar artificialmente a importância de certos padrões.

### 6a. Atributos Não Necessários (Colunas Constantes)

Se uma feature tem sempre o mesmo valor pra todos os exemplos, ela não contribui em nada — é peso morto no modelo. Vou verificar.

In [ ]:
X_clean, feature_names_clean, removed = remove_constant_columns(X_train, feature_names)

if removed:
    print(f'Atributos removidos (variância zero):')
    for name in removed:
        print(f'  - {name}')
else:
    print('Nenhum atributo constante encontrado.')
    print('Ou seja: todas as features têm variabilidade nos dados de treino.')
    print('Todas têm potencial pra contribuir na predição.')

print(f'\nFeatures mantidas: {len(feature_names_clean)} de {len(feature_names)}')

### 6b. Exemplos Não Necessários (Duplicatas)

Duplicatas são um problema sutil: se o mesmo paciente aparece 3 vezes no treino, o algoritmo vai dar 3 vezes mais peso pra ele. Isso não reflete a distribuição real dos dados e pode enviesar o modelo.

In [ ]:
X_dedup, y_dedup, n_dup = remove_duplicate_rows(X_clean, y_train)

if n_dup > 0:
    print(f'Exemplos duplicados removidos: {n_dup}')
else:
    print('Nenhum exemplo duplicado encontrado.')

print(f'Total antes: {len(y_train)} | Total após: {len(y_dedup)}')
print(f'\nPor que remover duplicatas? Elas inflam artificialmente a importância')
print(f'de certos padrões e podem enviesar o modelo. Removendo, cada exemplo')
print(f'contribui igualmente pro aprendizado.')

In [ ]:
X_train_clean = X_dedup
y_train_clean = y_dedup

## Etapa 7 — Análise de Desbalanceamento

Depois da limpeza, confiro a proporção entre as classes. Se tiver MUITO desbalanceado (tipo 10:1), eu precisaria aplicar SMOTE (gerar exemplos sintéticos da classe minoritária) ou undersampling (descartar exemplos da classe majoritária).

Mas sinceramente? Torço pra estar balanceado. Técnica de balanceamento artificial sempre introduz algum viés — SMOTE pode criar ruído sintético, undersampling joga informação fora.

In [ ]:
counts = count_target(y_train_clean)
total = sum(counts.values())

print(f'Distribuição atual (após limpeza):')
print(f'  Benigno (0): {counts[0]} ({format_percent(counts[0], total)})')
print(f'  Maligno  (1): {counts[1]} ({format_percent(counts[1], total)})')

majority_class = 0 if counts[0] > counts[1] else 1
minority_class = 1 - majority_class
ratio = counts[majority_class] / counts[minority_class]

print(f'\nRazão de desbalanceamento: {ratio:.2f}:1')
print(f'Classe majoritária: {majority_class} ({"Maligno" if majority_class == 1 else "Benigno"})')

print(f'\nMinha análise:')
if ratio < 2.0:
    print(f'  ✅ Dataset balanceado (razão < 2:1). Não vou aplicar SMOTE nem undersampling.')
    print(f'  Com uma razão tão próxima de 1:1, qualquer intervenção artificial faria')
    print(f'  mais mal do que bem. O baseline de classe majoritária já serve como')
    print(f'  referência pra avaliar se os modelos estão realmente aprendendo.')
else:
    print(f'  ⚠️ Dataset desbalanceado. Preciso aplicar técnica de balanceamento.')

## Etapa 8 — Limpeza de Dados

Agora vou analisar outliers, dados inconsistentes, redundantes e ausentes. Cada caso exige uma decisão diferente — não dá pra tratar tudo igual.

### 8a. Outliers (Método IQR)

Uso o método do Intervalo Interquartil (IQR) pra detectar outliers. A fórmula é clássica: tudo que estiver abaixo de Q1 - 1.5×IQR ou acima de Q3 + 1.5×IQR é considerado outlier.

Mas aqui vem uma decisão importante: **eu decidi MANTER todos os outliers**. Por quê? Porque os valores estão numa escala de 1 a 10 — são notas dadas por patologistas. Uma nota 10 não é um "erro de medição", é uma avaliação clínica válida que indica alta severidade. Se eu removesse os outliers, estaria jogando fora justamente os casos mais informativos pra detecção de câncer. No contexto médico, outliers são **sinais**, não ruído.

In [ ]:
n_outliers_total = 0
for col_idx, name in enumerate(feature_names_clean):
    col_vals = []
    for row in X_train_clean:
        val = row[col_idx]
        if val is not None:
            col_vals.append(val)
    lower, upper, outliers_idx, outliers_val = detect_outliers_iqr(col_vals)
    if outliers_val:
        print(f'{name:<32} {len(outliers_val):>3} outliers  (limites: [{lower:.2f}, {upper:.2f}])')
        n_outliers_total += len(outliers_val)
    else:
        print(f'{name:<32}    0 outliers')

print(f'\nTotal de outliers detectados: {n_outliers_total}')
print(f'Minha decisão: MANTER os outliers.')
print(f'Motivo: Os valores estão em escala de 1-10 (notas de patologistas).')
print(f'Um valor extremo (ex: 10) não é um erro — é uma avaliação clínica')
print(f'válida indicando alta severidade. Removê-los eliminaria justamente')
print(f'os casos mais informativos pra detecção de câncer.')

### 8b. Dados Inconsistentes

Verifico se tem valor fora do intervalo esperado [1, 10]. Se aparecesse um 11 ou um 0, seria erro de coleta — o patologista não dá nota 0 ou 11 numa escala de 1 a 10.

In [ ]:
n_inconsistent = 0
for col_idx, name in enumerate(feature_names_clean):
    for row_idx, row in enumerate(X_train_clean):
        val = row[col_idx]
        if val is not None and (val < FEATURE_MIN or val > FEATURE_MAX):
            n_inconsistent += 1
            print(f'Valor {val} no atributo {name} (linha {row_idx}) — fora de [1,10]')

if n_inconsistent == 0:
    print('Nenhum valor fora do intervalo esperado [1, 10].')
    print('Todos os atributos são notas inteiras de patologistas nessa escala, então')
    print('faz sentido que não tenha nenhum valor inválido.')

### 8c. Dados Redundantes

Duplicatas já foram tratadas na Etapa 6. Atributos com correlação cruzada > 0.95 seriam candidatos a remoção (se duas features são quase idênticas, manter as duas é redundante). Mas isso entra na Etapa 10 com a análise completa de correlação.

In [ ]:
print('Análise já realizada na Etapa 6 (remoção de duplicatas).')
print('Atributos redundantes seriam detectados por correlação cruzada > 0.95.')
print('A análise de correlação completa vem na Etapa 10.')

### 8d. Dados Ausentes

Verifico se sobrou algum valor None no treino. As 16 linhas problemáticas (Bare_nuclei vazio) já foram filtradas no carregamento inicial pelo mask.

Caso eu precise preencher algum valor, minha estratégia é usar a **MODA**, não a média. Por quê? Porque os atributos são discretos (notas inteiras de 1 a 10). Usar a média geraria valores decimais que não existem na prática clínica — um patologista nunca dá nota 3.7. A moda, por outro lado, sempre retorna um valor que realmente aparece nos dados.

In [ ]:
missing_cols = []
for col_idx, name in enumerate(feature_names_clean):
    n_missing = sum(1 for row in X_train_clean if row[col_idx] is None)
    if n_missing > 0:
        missing_cols.append((col_idx, name, n_missing))
        print(f'{name:<32} {n_missing} valores ausentes')

if not missing_cols:
    print('Nenhum valor ausente encontrado no conjunto de treino.')
    print('As 16 linhas com Bare_nuclei vazio já foram filtradas pelo mask na Etapa 1.')
    X_train_imputed = [list(row) for row in X_train_clean]
else:
    # Aplica imputação pela moda
    X_train_imputed = [list(row) for row in X_train_clean]
    for col_idx, name, n_missing in missing_cols:
        col_vals = [row[col_idx] for row in X_train_clean if row[col_idx] is not None]
        modes = calc_mode(col_vals)
        fill_value = modes[0]
        for row in X_train_imputed:
            if row[col_idx] is None:
                row[col_idx] = fill_value
        print(f'\n→ {name}: preenchidos {n_missing} valores com MODA = {int(fill_value)}')

    print(f'\nPor que moda e não média? Os atributos são pontuações clínicas')
    print(f'DISCRETAS (1-10). A média geraria valores decimais que não existem')
    print(f'na prática clínica. A moda mantém a integridade do domínio.')

### Imputação no Conjunto de Teste

Aplico os mesmos critérios de preenchimento no teste, sempre usando parâmetros calculados **no treino**. Isso é essencial pra evitar data leakage — se eu calculasse a moda no teste, estaria usando informação que o modelo não deveria ter acesso.

In [ ]:
# Identifica colunas com None no teste e preenche com moda do treino
missing_cols_test = []
for col_idx in range(len(feature_names_clean)):
    if any(row[col_idx] is None for row in X_test):
        col_vals_train = [row[col_idx] for row in X_train_imputed]
        modes = calc_mode(col_vals_train)
        fill_value = modes[0]
        missing_cols_test.append((col_idx, fill_value))

X_test_imputed = [list(row) for row in X_test]
for col_idx, fill_value in missing_cols_test:
    for row in X_test_imputed:
        if row[col_idx] is None:
            row[col_idx] = fill_value

print(f'Teste imputado: {len(missing_cols_test)} colunas corrigidas')

## Etapa 9 — Conversão e Normalização

Primeiro confiro se preciso converter algum tipo de dado (spoiler: não). Depois normalizo tudo com Min-Max [0, 1].

### 9a. Conversão de Tipos

Todos os atributos já são numéricos. A única conversão que fiz foi no target (2/4 → 0/1) lá na Etapa 1. Não tem atributo categórico, ordinal ou nominal que precise de one-hot encoding ou label encoding.

In [ ]:
print('Análise: Todos os atributos já são numéricos (quantitativos discretos).')
print('Não tem atributo simbólico, ordinal ou nominal pra converter.')
print('A conversão do target (2/4 → 0/1) já foi feita na Etapa 1.')

### 9b. Normalização Min-Max [0, 1]

Aplico Min-Max em todas as features. Os parâmetros (min, max) são calculados **só no treino** e depois aplicados no teste — isso é essencial pra evitar data leakage. Se eu normalizasse tudo junto, informação do teste vazaria pro treino.

Escolhi Min-Max em vez de Z-score por dois motivos:
1. **K-NN:** sem normalização, features com maior magnitude dominariam a distância euclidiana
2. **MLP:** valores muito grandes saturam a função sigmoide nos neurônios de entrada

Como a escala original já é limitada (1-10), o Min-Max preserva a estrutura da distribuição.

In [ ]:
# Calcula parâmetros no treino
X_train_norm, mins, maxs = normalize_minmax(X_train_imputed)

# Aplica os MESMOS parâmetros no teste (sem data leakage!)
X_test_norm = []
for row in X_test_imputed:
    new_row = []
    for col_idx, val in enumerate(row):
        denom = maxs[col_idx] - mins[col_idx]
        if denom == 0:
            new_row.append(0.0)
        else:
            new_row.append((val - mins[col_idx]) / denom)
    X_test_norm.append(new_row)

print('Parâmetros calculados no TREINO (min, max por coluna):')
for i, name in enumerate(feature_names_clean):
    print(f'  {name:<32} min={mins[i]:>4.1f}  max={maxs[i]:>4.1f}')

print(f'\nVerificação pós-normalização (treino):')
for i, name in enumerate(feature_names_clean):
    col_vals = [row[i] for row in X_train_norm]
    print(f'  {name:<32} min={min(col_vals):.4f}  max={max(col_vals):.4f}')

print(f'\nPor que Min-Max e não Z-score?')
print(f'  - Min-Max preserva a estrutura da distribuição original (1-10).')
print(f'  - Essencial pra K-NN: evita que features dominem a distância euclidiana.')
print(f'  - Essencial pra MLP: evita saturação das funções de ativação.')
print(f'  - Parâmetros do treino aplicados no teste → sem data leakage.')

## Etapa 10 — Redução de Dimensionalidade

Uso correlação de Pearson entre cada feature e o target. Features com correlação muito baixa (|r| < 0.10) são candidatas a remoção — se uma feature não tem relação linear com o target, ela provavelmente não ajuda na classificação.

Escolhi Pearson em vez de PCA por dois motivos:
1. Dá pra implementar do zero com Python puro (PCA exigiria decomposição em autovalores/autovetores, bem mais complexa)
2. A interpretação é direta — cada coeficiente te diz exatamente o quanto aquela feature se relaciona com o target

In [ ]:
correlations = feature_target_correlations(X_train_norm, y_train_clean, feature_names_clean)

print(f'Correlação de Pearson (feature × target):')
print(f'{"Feature":<32} {"r":>8} {"|r|":>8} {"Interpretação"}')
print(f'{"-"*32} {"-"*8} {"-"*8} {"-"*25}')

for name, r, abs_r in correlations:
    if abs_r < 0.10:
        interp = 'MUITO FRACA — candidata a remoção'
    elif abs_r < 0.30:
        interp = 'Fraca'
    elif abs_r < 0.50:
        interp = 'Moderada'
    elif abs_r < 0.70:
        interp = 'Forte'
    else:
        interp = 'MUITO FORTE'
    print(f'{name:<32} {r:>+8.4f} {abs_r:>8.4f} {interp}')

In [ ]:
# Decisão: remover features com |r| < 0.10
threshold = 0.10
to_remove = [name for name, r, abs_r in correlations if abs_r < threshold]

if to_remove:
    print(f'Decisão: REMOVER features com |r| < {threshold}.')
    remove_indices = [feature_names_clean.index(name) for name in to_remove]
    feature_names_final = [name for name in feature_names_clean if name not in to_remove]
    X_train_final = []
    for row in X_train_norm:
        X_train_final.append([row[i] for i in range(len(row)) if i not in remove_indices])
    X_test_final = []
    for row in X_test_norm:
        X_test_final.append([row[i] for i in range(len(row)) if i not in remove_indices])
    print(f'Features restantes: {len(feature_names_final)} de {len(feature_names_clean)}')
else:
    print(f'Decisão: MANTER todas as features.')
    print(f'Todas as features têm correlação relevante com o target (|r| ≥ {threshold}).')
    print(f'Até a Mitoses, que é a mais fraca, tem correlação moderada (|r| ≈ 0.36).')
    feature_names_final = list(feature_names_clean)
    X_train_final = X_train_norm
    X_test_final = X_test_norm

print(f'\nFeatures finais: {", ".join(feature_names_final)}')
print(f'\nPor que Pearson e não PCA?')
print(f'  - Pearson: implementável do zero, interpretação direta.')
print(f'  - PCA exigiria decomposição em autovalores/autovetores, bem mais complexa.')
print(f'  - Limiar |r| < 0.10 é conservador: remove apenas features realmente irrelevantes.')

## ✅ Fim do Notebook 2 — Resumo

Pré-processamento concluído! O que mudou:

| Métrica | Antes | Depois |
||---------|-------|--------|
|| Exemplos de treino | 546 | 362 |
|| Duplicatas | 184 | 0 |
|| Features | 9 | 9 (todas mantidas) |
|| Escala | 1-10 | [0, 1] (Min-Max) |
|| Balanceamento | ~1.9:1 | ~1.09:1 |

As variáveis `X_train_final`, `X_test_final`, `y_train_clean`, `y_test`, `feature_names_final` estão prontas pro **Notebook 3** (Modelagem Preditiva). Agora sim, bora treinar uns algoritmos!